# ECE 471 Anomaly Detection with EfficientAD
Jasper Halvorson and Liam Tanner


In [89]:
# Download the dataset, setup packages
import os
import cv2
import numpy as np
import numpy.typing as npt
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score

if not os.path.exists('dataset.zip'):
  !gdown 1_pRKXtYRjWjY0seYqyx25nOxjtr-mHYg
  !unzip -q -u dataset.zip
else:
  print('Already downloaded')

Already downloaded


In [90]:
# Some helper functions for your project
def load_dataset(class_name = 'pasta'):
  assert class_name in ['pasta', 'screws', 'capsule']
  dir = './dataset/'+class_name+'/'
  training_images = []
  testing_images = []
  testing_labels = []
  for file_name in os.listdir(dir+'train/good/'):
    training_images.append(cv2.cvtColor(cv2.imread(dir+'train/good/'+file_name), cv2.COLOR_BGR2RGB))
  for file_name in os.listdir(dir+'test/good/'):
    testing_images.append(cv2.cvtColor(cv2.imread(dir+'test/good/'+file_name), cv2.COLOR_BGR2RGB))
    testing_labels.append(0)
  for file_name in os.listdir(dir+'test/bad/'):
    testing_images.append(cv2.cvtColor(cv2.imread(dir+'test/bad/'+file_name), cv2.COLOR_BGR2RGB))
    testing_labels.append(1)

  # returns a normalized (0-1) numpy array of size (n,)
  return np.array(training_images)/255., np.array(testing_images)/255., np.array(testing_labels)

def basic_evaluation(predictions : np.ndarray, targets : np.ndarray):
  print(targets)
  print(predictions)
  print('AUROC Score:', roc_auc_score(targets, predictions))

In [91]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models
import torchvision
from fastai.data.external import untar_data, URLs
from torch.utils.data import DataLoader
from torchvision import transforms, datasets
import torch.optim as optim
import gdown

In [92]:
# Switch to GPU if possible
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [93]:
class AutoEncoder(nn.Module):
  def __init__(self,):
    super().__init__()

    # Encoder Layers as class attributes. Conv2d docs:
    # https://docs.pytorch.org/docs/stable/generated/torch.nn.Conv2d.html
    # Input channels is what the layer receives, and output channels is
    # specified in table 8 of the paper.
    # Kernel size, stride, and padding all taken directly from table 8
    self.EncConv1 = nn.Conv2d(in_channels=3, out_channels=32,
                              kernel_size=(4, 4), stride=(2, 2), padding=1)
    self.EncConv2 = nn.Conv2d(in_channels=32, out_channels=32,
                              kernel_size=(4, 4), stride=(2, 2), padding=1)
    self.EncConv3 = nn.Conv2d(in_channels=32, out_channels=64,
                              kernel_size=(4, 4), stride=(2, 2), padding=1)
    self.EncConv4 = nn.Conv2d(in_channels=64, out_channels=64,
                              kernel_size=(4, 4), stride=(2, 2), padding=1)
    self.EncConv5 = nn.Conv2d(in_channels=64, out_channels=64,
                              kernel_size=(4, 4), stride=(2, 2), padding=1)
    self.EncConv6 = nn.Conv2d(in_channels=64, out_channels=64,
                              kernel_size=(8, 8), stride=(1, 1), padding=0)

    # Decoder layers (more Conv2D)
    self.DecConv1 = nn.Conv2d(in_channels=64, out_channels=64,
                              kernel_size=(4, 4), stride=(1, 1), padding=2)
    self.DecConv2 = nn.Conv2d(in_channels=64, out_channels=64,
                              kernel_size=(4, 4), stride=(1, 1), padding=2)
    self.DecConv3 = nn.Conv2d(in_channels=64, out_channels=64,
                              kernel_size=(4, 4), stride=(1, 1), padding=2)
    self.DecConv4 = nn.Conv2d(in_channels=64, out_channels=64,
                              kernel_size=(4, 4), stride=(1, 1), padding=2)
    self.DecConv5 = nn.Conv2d(in_channels=64, out_channels=64,
                              kernel_size=(4, 4), stride=(1, 1), padding=2)
    self.DecConv6 = nn.Conv2d(in_channels=64, out_channels=64,
                              kernel_size=(4, 4), stride=(1, 1), padding=2)
    self.DecConv7 = nn.Conv2d(in_channels=64, out_channels=64,
                              kernel_size=(3, 3), stride=(1, 1), padding=1)
    self.DecConv8 = nn.Conv2d(in_channels=64, out_channels=384,
                              kernel_size=(3, 3), stride=(1, 1), padding=1)

    # Bilinear Layers: https://docs.pytorch.org/docs/stable/generated/torch.nn.Upsample.html
    self.Bilinear1 = nn.Upsample(size=(3, 3), mode="bilinear")
    self.Bilinear2 = nn.Upsample(size=(8, 8), mode="bilinear")
    self.Bilinear3 = nn.Upsample(size=(15, 15), mode="bilinear")
    self.Bilinear4 = nn.Upsample(size=(32, 32), mode="bilinear")
    self.Bilinear5 = nn.Upsample(size=(63, 63), mode="bilinear")
    self.Bilinear6 = nn.Upsample(size=(127, 127), mode="bilinear")
    self.Bilinear7 = nn.Upsample(size=(64, 64), mode="bilinear")

    # Dropout Layers: https://docs.pytorch.org/docs/stable/generated/torch.nn.Dropout.html
    self.Dropout1To6 = nn.Dropout(p=0.2) # All dropout layers have the same p value

    # Relu activation function (for convenience)
    self.relu = nn.ReLU()

  def forward(self, x):
    # Compute a forward pass. Here we just combine the layers in the order of table 8 in the EfficientAD paper
    out = self.relu(self.EncConv1(x))
    out = self.relu(self.EncConv2(out))
    out = self.relu(self.EncConv3(out))
    out = self.relu(self.EncConv4(out))
    out = self.relu(self.EncConv5(out))
    out = self.EncConv6(out)

    out = self.Bilinear1(out)

    out = self.relu(self.DecConv1(out))
    out = self.Dropout1To6(out)
    out = self.Bilinear2(out)

    out = self.relu(self.DecConv2(out))
    out = self.Dropout1To6(out)
    out = self.Bilinear3(out)

    out = self.relu(self.DecConv3(out))
    out = self.Dropout1To6(out)
    out = self.Bilinear4(out)

    out = self.relu(self.DecConv4(out))
    out = self.Dropout1To6(out)
    out = self.Bilinear5(out)

    out = self.relu(self.DecConv5(out))
    out = self.Dropout1To6(out)
    out = self.Bilinear6(out)

    out = self.relu(self.DecConv6(out))
    out = self.Dropout1To6(out)
    out = self.Bilinear7(out)

    out = self.relu(self.DecConv7(out))
    out = self.DecConv8(out)

    return out


In [94]:
class PDN(nn.Module):
  def __init__(self, out_channels=384, padding=True):
    super(PDN, self).__init__()

    p = 3 if padding else 0

    self.conv1 = nn.Conv2d(3, 128, kernel_size=4, stride=1, padding=p);
    self.avgpool1 = nn.AvgPool2d(kernel_size=2, stride=2, padding=1);
    self.conv2 = nn.Conv2d(128, 256, kernel_size=4, stride=1, padding=p);
    self.avgpool2 = nn.AvgPool2d(kernel_size=2, stride=2, padding=1);
    self.conv3 = nn.Conv2d(256, 256, kernel_size=3, stride=1, padding=1);
    self.conv4 = nn.Conv2d(256, out_channels, kernel_size=4, stride=1, padding=0);

    self.relu = nn.ReLU(inplace=True);

  # Compute the forward pass
  def forward(self, x):
    x = self.relu(self.conv1(x))
    x = self.avgpool1(x)
    x = self.relu(self.conv2(x))
    x = self.avgpool2(x)
    x = self.relu(self.conv3(x))
    x = self.conv4(x)
    return x

In [95]:
def image_preprocessing(images_np, device):
  img = torch.tensor(images_np, dtype=torch.float32)
  img = img.permute(0, 3, 1, 2).contiguous()
  img = torch.nn.functional.interpolate(img, size=(256, 256), mode='bilinear', align_corners=False)
  img = transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])(img)

  return img.to(device)

In [96]:
def download_if_needed(id, output):
  # if not os.path.exists(output) and id:
  gdown.download(id=id, output=output, quiet=False)

In [97]:
def get_models(dataset):
  # Setup (hardcoded) google drive IDs
  file_ids = {
      # Pasta model with increased learning rate and 10 000 iterations
      'pasta': {
          'student':     '1EMr8iPo6JOtiFh7HiIVCilGKMRTpCuTe',
          'autoencoder': '1nLz8ybBXltzGvpnmgfW85eGZkZt0jVuT',
          'metadata':    '1IP-1vUI9KNfDiwdfyFhZ32P3vkZUTO56'
      },
      # Pasta models with 70 000 iterations:
      # 'pasta': {
      #     'student':     '1ioO_6Zq-gqtcOhqmU0jIgEcSNK9NEBJ5',
      #     'autoencoder': '1cDzg5NrLvq_8WVr5sEWaU6aR3a3PRh-X',
      #     'metadata':    '1-AIsBdERCCFf5S9Ipm81JhG8VkK7_m4l'
      # },
      'screws': {
          'student':     '1WkOWq1LIvDRAEIJ_17aFvaazN4Y-7L47',
          'autoencoder': '1LEwBxBwRRcQ3nl6w1CLvk3IgnLgGqIB5',
          'metadata':    '1OT7xKCbfI4VS5PQb68mU_nXUuyT-qPHa'
      },
      'capsule': {
          'student':     '1RdzqHx8zgmI41Gi40yn2du5_FykeF8Rj',
          'autoencoder': '1Kgo_A0qFGP1V365Ly_SjFl4NptbiQZ0Z',
          'metadata':    '1kyZzR6X75C-jmF5GXmsMm37S7M9TBi9A'
      }
  }
  # Always download same teacher
  if dataset == 'pasta':
    download_if_needed(id='1TWO-YuEy2TpxhiPQ54la4_ApShQ-J6RY', output='teacher_pasta.pth')
  else:
    download_if_needed(id='1TWO-YuEy2TpxhiPQ54la4_ApShQ-J6RY', output='teacher.pth')
  # Download all assets for the specified dataset
  ids = file_ids[dataset]
  download_if_needed(ids['student'],     f'/content/{dataset}_student.pth')
  download_if_needed(ids['autoencoder'], f'/content/{dataset}_autoencoder.pth')
  download_if_needed(ids['metadata'],    f'/content/{dataset}_metadata.pth')

In [98]:
''' We include training functions here, but note that all other code required to train is otherwise omitted from this file '''
class AnomalyDetector:
    def __init__(self):
      self.teacher = None
      self.student = None
      self.autoencoder = None
      self.mu = None
      self.sigma = None
      self.st_quantile_90 = None
      self.st_quantile_995 = None
      self.sa_quantile_90 = None
      self.sa_quantile_995 = None

    def create_model(self, dataset: np.ndarray, dataset_name: str):
      # Download weights
      print(f"Downloading weights for {dataset_name}...")
      get_models(dataset_name)

      # Load metadata (mu, sigma, quantiles)
      metadata = torch.load(f'/content/{dataset_name}_metadata.pth', map_location=device)
      self.mu              = metadata['mu'].to(device)
      self.sigma           = metadata['sigma'].to(device)
      self.st_quantile_90  = metadata['st_90'].to(device)
      self.st_quantile_995 = metadata['st_995'].to(device)
      self.sa_quantile_90  = metadata['sa_90'].to(device)
      self.sa_quantile_995 = metadata['sa_995'].to(device)

      # Load teacher
      self.teacher = PDN().to(device)
      if dataset_name == 'pasta':
        self.teacher.load_state_dict(torch.load('/content/teacher_pasta.pth', map_location=device))
      else:
        self.teacher.load_state_dict(torch.load('/content/teacher.pth', map_location=device))
      self.teacher.eval()

      # Load student
      self.student = PDN(out_channels=768).to(device)
      self.student.load_state_dict(torch.load(f'/content/{dataset_name}_student.pth', map_location=device))
      self.student.eval()

      # Load autoencoder
      self.autoencoder = AutoEncoder().to(device)
      self.autoencoder.load_state_dict(torch.load(f'/content/{dataset_name}_autoencoder.pth', map_location=device))
      self.autoencoder.eval()

      print(f"Model for {dataset_name} loaded successfully.")

    def inference(self, test_image):
      '''Implementation of Algorithm 2 from the paper'''
      # Get test image and get outputs of each network
      teacher_output = self.teacher(test_image)
      student_output = self.student(test_image)
      autoencoder_output = self.autoencoder(test_image)

      # Normalize the teacher output and get the differences as in training function
      normalized_teacher_output = (teacher_output - self.mu) / self.sigma
      student_output_f384 = student_output[:, :384, :, :]
      student_output_l384 = student_output[:, 384:, :, :]
      diff_st = (normalized_teacher_output - student_output_f384)**2
      diff_sa = (autoencoder_output - student_output_l384)**2

      # Generate anomaly maps as done in training
      map_st = diff_st.mean(dim=1)
      map_sa = diff_sa.mean(dim=1)
      map_st = F.interpolate(input=map_st.unsqueeze(1), size=(256, 256), mode='bilinear')
      map_sa = F.interpolate(input=map_sa.unsqueeze(1), size=(256, 256), mode='bilinear')

      # Normalize the original maps and combine them
      normalized_map_st = 0.1 * (map_st - self.st_quantile_90) / (self.st_quantile_995 - self.st_quantile_90)
      normalized_map_sa = 0.1 * (map_sa - self.sa_quantile_90) / (self.sa_quantile_995 - self.sa_quantile_90)

      combined_map = 0.5 * normalized_map_st + 0.5 * normalized_map_sa
      image_level_score = torch.flatten(combined_map).max()

      return combined_map, image_level_score

    def predict(self, test_data: np.ndarray):
      scores = []
      anomaly_maps = []
      with torch.no_grad():
        for img in test_data:
          # img_tensor = HWC_to_NCHW(img).to(device)
          # img_tensor = image_preprocessing(img, device)
          anomaly_map, anomaly_score = self.inference(img)
          scores.append(float(anomaly_score.cpu()))
          anomaly_maps.append(anomaly_map.cpu())

      return np.array(anomaly_maps), np.array(scores)

    def train_student_and_autoencoder(self, training_data, validation_data, testing_data, testing_labels):
      # Follow Algorithm 1 in the paper to train the autoencoder and student
      # Need pretrained teacher (we have), sequence of training/validation images (given)
      # 1. Compute Mu and Sigma based on training data set (Algorithm 1, Steps 3-9)
      teacher_outputs = []
      with torch.no_grad():
        for i in range(len(training_data)):
          # Prepare image and get teacher features
          img = training_data[i:i+1]
          y_prime = self.teacher(img) # Shape: [1, 384, 64, 64]
          teacher_outputs.append(y_prime)

      # Concatenate all outputs into one large tensor [N, 384, 64, 64]
      all_outputs = torch.cat(teacher_outputs, dim=0)

      # Compute mu and sigma per channel across the entire dataset
      # We reduce over Dim 0 (Images), Dim 2 (Height), and Dim 3 (Width)
      self.mu = torch.mean(all_outputs, dim=(0, 2, 3), keepdim=True)
      self.sigma = torch.std(all_outputs, dim=(0, 2, 3), keepdim=True)

      # Now we initialize a joint Adam optimizer for parameters of student and autoencoder (learning rate and weight decay from algorithm 1)
      lr = 1e-4
      weight_decay = 1e-5
      sa_parameters = list(self.student.parameters()) + list(self.autoencoder.parameters())
      sa_optimizer = optim.Adam(sa_parameters, lr=lr, weight_decay=weight_decay)

      # Initialize pretraining iterator once, outside the main loop.
      # We don't want to create a new pretraining iterator instance with each loop iteration
      path = '' # In training this is set above after loading in imagenet data
      distill_dataset = datasets.ImageFolder(root=os.path.join(path, 'train'), transform=transform)
      distill_loader = DataLoader(distill_dataset, batch_size=1, shuffle=True) # add this because it is loading whole thing in teacher now
      pretrain_iter = iter(distill_loader)

      batch_size = min(8, len(training_data))

      # Main training loop
      for i in range(1, 70001):
        # Choose 'random' training image, do fwd pass with teacher, do fwd pass with student.
        indices = torch.randint(0, len(training_data), (batch_size,), device=training_data.device)
        random_image = training_data[indices]

        with torch.no_grad():
          teacher_output = self.teacher(random_image)
        student_output = self.student(random_image)

        # Get the normalized teacher output per channel
        normalized_teacher_output = (teacher_output - self.mu) / self.sigma

        # Now compute square difference between normalized teacher output and the first 384 entries of the student output
        student_output_f384 = student_output[:, :384, :, :]
        diff_st = (normalized_teacher_output - student_output_f384)**2

        # Get the 0.999 quantile of the difference, and compute loss as mean of diff_st entries over or equal to the quantile
        diff_st_999 = torch.quantile(torch.flatten(diff_st), q=0.999)

        # Pytorch boolean indexing rather than slow python loop
        loss_hard = diff_st[diff_st >= diff_st_999].mean()

        # Now choose a random pretraining ImageNet image. Load pretraining images more efficiently by
        # creating an infinite loop of the image data (so that we never run out) with a try block
        # This way we don't need to create a new pretraining iterator with each loop iteration
        try:
          pretraining_image, _ = next(pretrain_iter)
        except StopIteration:
          pretrain_iter = iter(distill_loader)
          pretraining_image, _ = next(pretrain_iter)
        pretraining_image = pretraining_image.to(device)

        # Now get the student teacher loss with hard loss and a regularization term defined in the algorithm
        student_output_pretaining_384 = self.student(pretraining_image)[:, :384, :, :]
        reg_term = (student_output_pretaining_384**2).mean()
        loss_st = loss_hard + reg_term

        # Now we augment the training images for use on the encoder branch. We do this as we want the encoder to
        # learn more general structures. (Use adjustments from Algorithm 1)
        augmentation_index = np.random.randint(1, 4)
        lam = np.random.uniform(0.8, 1.2)
        if augmentation_index == 1:
          augmented_images = torchvision.transforms.functional.adjust_brightness(random_image, lam)
        elif augmentation_index == 2:
          augmented_images = torchvision.transforms.functional.adjust_contrast(random_image, lam)
        elif augmentation_index == 3:
          augmented_images = torchvision.transforms.functional.adjust_saturation(random_image, lam)

        # With the augmented training images, run forward passes on the autoencoder and the teacher,
        # and normalize as we did before.
        autoencoder_output = self.autoencoder(augmented_images)
        with torch.no_grad():
          teacher_output_augmented = self.teacher(augmented_images)

        normalized_teacher_output_augmented = (teacher_output_augmented - self.mu) / self.sigma
        student_output_augmented = self.student(augmented_images)
        student_output_augmented_l384 = student_output_augmented[:, 384:, :, :]
        diff_ta = (normalized_teacher_output_augmented - autoencoder_output)**2
        diff_sa = (autoencoder_output - student_output_augmented_l384)**2

        # For the loss, we take the sum of the means of each of the differences (teacher-autoencoder, student-autoencoder,
        # and the student-teacher loss from earlier)
        loss_ta = torch.mean(diff_ta)
        loss_sa = torch.mean(diff_sa)
        loss = loss_st + loss_ta + loss_sa

        if i % 500 == 0:
          print("At iteration:", i, " loss: ",  loss)

        # Update the Adam optimizer parameters. Clear old gradients (avoid accumulation), and step
        sa_optimizer.zero_grad()
        loss.backward()
        sa_optimizer.step()

        # Decay learning rate after 4500 iterations
        if i == 66500:
          sa_optimizer.param_groups[0]['lr'] = 1e-5


      # Here, the training of student and autoencoder weights is done. We now make anomaly maps
      # and get anomaly scores
      st_anomaly_scores = []
      sa_anomaly_scores = []

      self.student.eval()
      self.autoencoder.eval()
      # Use more normal images from the validation set to get high end differences in normal images
      for i in range(len(validation_data)):
        val_image = validation_data[i:i+1]

        # Get outputs and normalize/compute loss similar to before
        with torch.no_grad():
          teacher_output = self.teacher(val_image)
        student_output = self.student(val_image)
        autoencoder_output = self.autoencoder(val_image)

        student_output_f384 = student_output[:, :384, :, :]
        student_output_l384 = student_output[:, 384:, :, :]
        normalized_teacher_output = (teacher_output - self.mu) / self.sigma
        diff_st = (normalized_teacher_output - student_output_f384)**2
        diff_sa = (autoencoder_output - student_output_l384)**2

        # Make anomaly maps using the differences. Take mean difference across channel for each output pixel
        map_st = diff_st.mean(dim=1)
        map_sa = diff_sa.mean(dim=1)

        # Resize with bilinear interpolation (like done in the autoencoder forward pass)
        map_st = F.interpolate(input=map_st.unsqueeze(1), size=(256, 256), mode='bilinear')
        map_sa = F.interpolate(input=map_sa.unsqueeze(1), size=(256, 256), mode='bilinear')

        # Append vectorized anomaly scores to score lists
        st_anomaly_scores.append(torch.flatten(map_st))
        sa_anomaly_scores.append(torch.flatten(map_sa))

      # We now have a list of flattened local (st) anomaly maps, and global (sa)
      # Get quantile values for 0.9 and 0.995 for each map so we have a reference later for what to consider an anomaly
      self.st_quantile_90 = torch.quantile(torch.cat(st_anomaly_scores), 0.90)
      self.st_quantile_995 = torch.quantile(torch.cat(st_anomaly_scores), 0.995)

      self.sa_quantile_90 = torch.quantile(torch.cat(sa_anomaly_scores), 0.90)
      self.sa_quantile_995 = torch.quantile(torch.cat(sa_anomaly_scores), 0.995)

      return self.teacher, self.student, self.autoencoder, self.mu, self.sigma, self.st_quantile_90, self.st_quantile_995, self.sa_quantile_90, self.sa_quantile_995

    def train_teacher(self, oracle_extractor, distill_images, oracle_mean, oracle_variance, iterations=60000, batch_size=16):
      self.teacher.train()
      oracle_extractor.eval()

      # Algorithm 3, Line 13: Adam with lr=1e-4 and weight_decay=1e-5
      optimizer = optim.Adam(self.teacher.parameters(), lr=1e-4, weight_decay=1e-5)

      # Normalize oracle output (Algorithm 3, Line 22)
      self.mu = oracle_mean.view(1, -1, 1, 1).to(device)
      self.sigma = oracle_variance.view(1, -1, 1, 1).to(device)

      # Algorithm 3, Lines 14-30: Main training loop
      for iteration in range(1, iterations + 1):
        # Algorithm 3, Lines 16-27: Inner batch loop of 16 images
        # We dont actually need to loop like is done in the algorithm, can process the whole batch at once
        indices = torch.randint(0, len(distill_images), (batch_size,), device=distill_images.device)
        images = distill_images[indices]

        # Algorithm 3, Line 18: Convert to grayscale with probability 0.1
        # Apply this to batch all at once using a mask
        gray = transforms.functional.rgb_to_grayscale(images, num_output_channels=3)
        gray_mask = (torch.rand(images.shape[0], 1, 1,1, device=device) < 0.1)
        images = torch.where(gray_mask, gray, images)

        # Algorithm 3, Line 19: Images are already resized to 256x256 by the dataloader
        # but we need a 512x512 version for the oracle
        images_oracle = F.interpolate(images, size=(512, 512), mode='bilinear', align_corners=False)

        # Algorithm 3, Lines 21-22: Get normalized oracle features
        with torch.no_grad():
          y_oracle = oracle_extractor(images_oracle)
          y_oracle_normalized = (y_oracle - self.mu) / self.sigma

        # Algorithm 3, Line 23: Get teacher features on 256x256 image
        y_teacher = self.teacher(images)

        # Algorithm 3, Lines 24-28: Compute squared difference and mean loss
        D_dist = (y_oracle_normalized - y_teacher) ** 2
        L_batch = D_dist.mean()
        # End of Pseudocode loop over batch

        # Algorithm 3, Line 29: Update teacher parameters
        optimizer.zero_grad()
        L_batch.backward()
        optimizer.step()
        if iteration % 500 == 0:
          print(f"Iteration {iteration}/{iterations} | Loss: {L_batch.item():.6f}")


In [99]:
# Get dataset statistics to save and use later
transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])


In [100]:
get_models('screws')

Downloading...
From: https://drive.google.com/uc?id=1TWO-YuEy2TpxhiPQ54la4_ApShQ-J6RY
To: /content/teacher.pth
100%|██████████| 10.8M/10.8M [00:00<00:00, 36.7MB/s]
Downloading...
From: https://drive.google.com/uc?id=1WkOWq1LIvDRAEIJ_17aFvaazN4Y-7L47
To: /content/screws_student.pth
100%|██████████| 17.1M/17.1M [00:00<00:00, 46.1MB/s]
Downloading...
From: https://drive.google.com/uc?id=1LEwBxBwRRcQ3nl6w1CLvk3IgnLgGqIB5
To: /content/screws_autoencoder.pth
100%|██████████| 4.40M/4.40M [00:00<00:00, 34.3MB/s]
Downloading...
From: https://drive.google.com/uc?id=1OT7xKCbfI4VS5PQb68mU_nXUuyT-qPHa
To: /content/screws_metadata.pth
100%|██████████| 6.14k/6.14k [00:00<00:00, 9.21MB/s]


In [ ]:
get_models('pasta')

Downloading...
From: https://drive.google.com/uc?id=1TWO-YuEy2TpxhiPQ54la4_ApShQ-J6RY
To: /content/teacher_pasta.pth
100%|██████████| 10.8M/10.8M [00:00<00:00, 41.7MB/s]
Downloading...
From: https://drive.google.com/uc?id=1EMr8iPo6JOtiFh7HiIVCilGKMRTpCuTe
To: /content/pasta_student.pth
100%|██████████| 17.1M/17.1M [00:00<00:00, 174MB/s]
Downloading...
From: https://drive.google.com/uc?id=1nLz8ybBXltzGvpnmgfW85eGZkZt0jVuT
To: /content/pasta_autoencoder.pth
100%|██████████| 4.40M/4.40M [00:00<00:00, 128MB/s]


In [ ]:
get_models('capsule')

In [ ]:
def visual_results(test_images, anomaly_maps, scores, true_labels, threshold):
  # Visualization of results
  predicted_classes = (scores >= threshold).astype(int)

  for i in range(len(scores)):
    img = cv2.resize(test_images[i], (256, 256))
    plt.imshow(img)
    im = plt.imshow(np.squeeze(anomaly_maps[i]), cmap='jet', alpha=0.5)
    plt.colorbar(im, fraction=0.046, pad=0.04, label='Anomaly Intensity')
    plt.title(f"Anomaly Map - {true_labels[i]}\nScore: {scores[i].item():.4f}\nPrediction: {int(scores[i]>=threshold)}")
    plt.axis('off')
    plt.show()

In [ ]:
def evaluate_results(scores, testing_labels, anomaly_threshold):
  print("scores: ", scores)
  predictions = (scores >= anomaly_threshold).astype(int)
  auroc = roc_auc_score(testing_labels, scores)
  accuracy = accuracy_score(testing_labels, predictions)
  f1 = f1_score(testing_labels, predictions)

  return predictions, auroc, accuracy, f1

In [ ]:
def do_analysis(ad, class_name):
  training_images, testing_images, testing_labels = load_dataset(class_name=class_name)
  test_images = image_preprocessing(testing_images, device)
  train_images = image_preprocessing(training_images, device)
  ad.create_model(train_images, class_name)
  _, normal_scores = ad.predict(train_images)
  # thres = np.quantile(normal_scores, 0.9)
  # Set threshold based on dataset
  thresholds = {
      'pasta':   0.19,
      'screws':  0.131,
      'capsule': 0.198
  }
  thres = thresholds[class_name]
  anomaly_maps, scores = ad.predict(test_images)
  visual_results(test_images, anomaly_maps, scores, testing_labels, thres)
  predictions, auroc, accuracy, f1 = evaluate_results(scores, testing_labels, thres)

  print("AUROC: ", auroc)
  print("F1: ", f1)
  print("Accuracy: ", accuracy)
  print("Predictions: ", predictions)
  print("Test Labels: ", testing_labels)


In [ ]:
do_analysis(AnomalyDetector(), 'pasta')

In [ ]:
do_analysis(AnomalyDetector(), 'screws')

In [ ]:
do_analysis(AnomalyDetector(), 'capsule')